# Mini attention: solving key-value retrieval

Notebook companion to [`../examples/03_mini_attention_retrieval.py`](../examples/03_mini_attention_retrieval.py) —
same code, same math, walked through cell by cell instead of read top to bottom
as one file. It builds part-1-maths.html §11, "attention built in five steps":
score relevance &rarr; separate Q from K &rarr; softmax into weights &rarr;
V-weighted retrieval &rarr; scale by 1/&radic;d<sub>k</sub>. No autograd, no ML
framework — every forward step is undone by hand in `backward()`, and a
numerical gradient check at the end proves the hand-derived formulas are
correct.

**The task.** Single-head scaled dot-product attention solving toy key-value
retrieval: given `n_pairs` (key, value) pairs and a query key, find the pair
whose key matches the query and return its value. This is attention doing the
one thing it is for — a content-based lookup — with nothing else (no stacked
blocks, no causal mask, no multi-head) in the way.

**Why key and value share one sequence position.** Each (key, value) pair
occupies *one* position, not two — the more obvious `[k1 v1 k2 v2 ... QUERY]`
layout, one token per key and a separate token per value, does not work here.
In that layout the value answering a query sits at a *different* position
than the key identifying it (key<sub>i</sub> at position 2i, value<sub>i</sub>
at position 2i+1). A single attention layer computes one V per position from
that position's *own* content — it cannot first locate the matching key and
then hop one position over to read the value; that "find, then shift" is a
two-step (two-layer) induction-head circuit, not a one-step lookup.
Empirically, training the interleaved layout end-to-end never beats chance
(attention stayed at uniform weight over ~1500 gradient updates across every
learning rate, batch size, and initialization tried). Binding a pair's key
*and* value to the same position turns the task back into what §11 actually
derives: one dot product decides relevance, one V delivers the payload — a
single hop, exactly what softmax(QK<sup>T</sup>/&radic;d<sub>k</sub>)V
computes.

In [1]:
import numpy as np

## The task generator

Each sample is `n_pairs` distinct keys (no duplicates, so exactly one key
matches the query — the task is well-posed) each bound to a distinct value,
followed by one query position holding one of those keys and no value. The
target is the value bound to that key.

Concretely, for `n_pairs=3` one sample might look like:

```
position:   0      1      2      3 (query)
key:        k4     k1     k5     k1
value:      v2     v0     v5     (none)
```

The key at the query slot (`k1`) matches position 1, so the target value is
`v0`, and attention *should* end up concentrated on position 1.

Values are drawn without replacement too. If they could repeat, "always
output the most common value in the sequence" becomes a shortcut that scores
~45% without ever looking at the query — the model can hide in that local
minimum and never learn to attend at all. Distinct values remove the
shortcut: the only way above chance (1/n_pairs) is genuine content-based
retrieval.

`seq_len_for` and `make_retrieval_task` below are copied verbatim from the
`.py` file.

In [2]:
def seq_len_for(n_pairs):
    return n_pairs + 1  # one position per (key,value) pair + one query position


def make_retrieval_task(n_pairs=4, key_vocab=6, value_vocab=6, n_samples=2000, seed=0):
    """Generate the toy retrieval task.

    Each sample: n_pairs distinct keys (no duplicates -> the task is
    well-posed, exactly one key matches the query) each bound to a distinct
    value (see note above), followed by a query position holding one of the
    n_pairs keys and no value. Target: the value bound to that key.

    Returns
    -------
    key_ids   : (n_samples, seq_len) int array -- key id at every position
                (query slot holds the query key)
    val_ids   : (n_samples, seq_len) int array -- value id at every position
                (query slot holds the sentinel index `value_vocab`, meaning
                "no value here")
    targets   : (n_samples,) int array, raw value-vocab index (0..value_vocab-1)
    match_pos : (n_samples,) int array, the sequence position whose key
                matches the query -- i.e. where attention *should*
                concentrate. Not used in training; only for inspecting
                attention afterwards.
    """
    assert key_vocab >= n_pairs, "need >= n_pairs distinct keys to avoid duplicates"
    assert value_vocab >= n_pairs, "need >= n_pairs distinct values to remove the mode shortcut"
    rng = np.random.default_rng(seed)
    L = seq_len_for(n_pairs)
    NO_VALUE = value_vocab  # sentinel row in E_val: "this position carries no value"

    key_ids = np.empty((n_samples, L), dtype=np.int64)
    val_ids = np.empty((n_samples, L), dtype=np.int64)
    targets = np.empty(n_samples, dtype=np.int64)
    match_pos = np.empty(n_samples, dtype=np.int64)

    for i in range(n_samples):
        keys = rng.choice(key_vocab, size=n_pairs, replace=False)
        values = rng.choice(value_vocab, size=n_pairs, replace=False)
        q_idx = rng.integers(0, n_pairs)                # which pair is queried

        key_ids[i, :n_pairs] = keys
        val_ids[i, :n_pairs] = values
        key_ids[i, n_pairs] = keys[q_idx]                # query: same key id as its pair
        val_ids[i, n_pairs] = NO_VALUE
        targets[i] = values[q_idx]
        match_pos[i] = q_idx

    return key_ids, val_ids, targets, match_pos


# quick look at one sample
_k, _v, _t, _m = make_retrieval_task(n_pairs=3, key_vocab=6, value_vocab=6, n_samples=1, seed=7)
print("key_ids  ", _k[0])
print("val_ids  ", _v[0], " (6 = sentinel 'no value')")
print("target   ", _t[0], " <- value bound to the queried key")
print("match_pos", _m[0], " <- position attention should land on")

key_ids   [3 4 5 3]
val_ids   [4 1 3 6]  (6 = sentinel 'no value')
target    4  <- value bound to the queried key
match_pos 0  <- position attention should land on


## Parameters

Three embedding tables (key, value, position — added together, per the token
scheme above) plus the three attention projections W<sub>q</sub>, W<sub>k</sub>,
W<sub>v</sub> and an output projection W<sub>o</sub>, b<sub>o</sub> mapping the
retrieved context back to a distribution over the value vocabulary. Single
head, so d<sub>k</sub> = d_model — no subspace split. Projection weights are
scaled by 1/&radic;fan-in (the same "sum of n independent products, spread
grows like &radic;n" reasoning §9/§11 use for initialization), embeddings just
get small Gaussian noise.

In [3]:
def init_params(seed=0, d_model=16, n_pairs=4, key_vocab=6, value_vocab=6):
    """Small random init. d_k = d_model (single head, no subspace split)."""
    rng = np.random.default_rng(seed)
    L = seq_len_for(n_pairs)

    def small(*shape):
        return rng.normal(0, 1.0 / np.sqrt(shape[-1]), shape)

    params = {
        "E_key": rng.normal(0, 0.1, (key_vocab, d_model)),          # key embedding
        "E_val": rng.normal(0, 0.1, (value_vocab + 1, d_model)),     # value embedding (+1: "no value" sentinel)
        "E_pos": rng.normal(0, 0.1, (L, d_model)),                   # positional embedding
        "Wq": small(d_model, d_model),
        "Wk": small(d_model, d_model),
        "Wv": small(d_model, d_model),
        "Wo": small(value_vocab, d_model),                           # (out, in), matches part-2's convention
        "bo": np.zeros(value_vocab),
        "meta": {
            "d_model": d_model, "n_pairs": n_pairs,
            "key_vocab": key_vocab, "value_vocab": value_vocab, "seq_len": L,
        },
    }
    return params

## Forward: the five steps of §11, with a single query row

Only the attention output *at the query position* (the last slot) is ever
read downstream, so instead of computing a full L&times;L self-attention
matrix, one query vector is scored against all L keys — same five steps,
same math, one row of the QK<sup>T</sup> matrix.

- **Embed.** `X = E_key[key_ids] + E_val[val_ids] + E_pos` — key identity +
  value payload + position, summed into one vector per pair. `x_query` is
  just the last row of X.
- **Step 2 — separate "what I seek" from "what I am".** `Q = x_query @ Wq`
  (one query vector per sample), `K = X @ Wk` (every position advertises
  itself). `V = X @ Wv` is computed here too so it's ready for the weighted
  sum in step 4.
- **Step 1 — score relevance.** `scores = einsum("bd,bld->bl", Q, K)` — the
  dot product of the query against every position's key.
- **Step 5 — scale.** divide by &radic;d<sub>k</sub> so softmax doesn't
  saturate (part-1 §11's variance argument: a sum of d<sub>k</sub> roughly
  independent unit-variance products has variance d<sub>k</sub>, so typical
  size &asymp; &radic;d<sub>k</sub>).
- **Step 3 — turn scores into weights.** `weights = softmax(scores)` along
  the sequence axis — rows sum to 1.
- **Step 4 — retrieve.** `context = einsum("bl,bld->bd", weights, V)` — the
  weighted sum of V that is actually retrieved. `logits = context @ Wo.T + bo`
  turns that context into a distribution over the value vocabulary.

Everything needed for backward — X, Q, K, V, weights, context, probs — is
kept in `cache`.

In [4]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)   # log-sum-exp trick: no overflow, same result
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def cross_entropy_loss(probs, target):
    B = target.shape[0]
    return -np.mean(np.log(probs[np.arange(B), target] + 1e-12))


def forward(key_ids, val_ids, params):
    """key_ids, val_ids: (B, L) int arrays. Returns logits, attn weights, cache."""
    if key_ids.ndim == 1:
        key_ids = key_ids[None, :]
        val_ids = val_ids[None, :]
    B, L = key_ids.shape
    E_key, E_val, E_pos = params["E_key"], params["E_val"], params["E_pos"]
    Wq, Wk, Wv, Wo, bo = params["Wq"], params["Wk"], params["Wv"], params["Wo"], params["bo"]
    d_k = Wq.shape[1]

    X = E_key[key_ids] + E_val[val_ids] + E_pos[None, :, :]   # (B,L,d): key + value + position, one vector per pair
    x_query = X[:, -1, :]                               # (B,d): the last slot IS the query

    # Step 2 -- separate "what I seek" (Q) from "what I am" (K)
    Q = x_query @ Wq                                    # (B,d_k)   -- one query vector per sample
    K = X @ Wk                                           # (B,L,d_k) -- every position advertises itself
    # Step 4 (payload projection, computed here so it's ready for the weighted sum)
    V = X @ Wv                                           # (B,L,d_v)

    # Step 1 -- score relevance: dot product of query against every key
    # Step 5 -- scale by 1/sqrt(d_k) so softmax doesn't saturate
    scores = np.einsum("bd,bld->bl", Q, K) / np.sqrt(d_k)   # (B,L)

    # Step 3 -- turn scores into weights
    weights = softmax(scores, axis=-1)                   # (B,L), rows sum to 1

    # Step 4 -- weighted sum of V: what actually gets retrieved
    context = np.einsum("bl,bld->bd", weights, V)          # (B,d_v)

    logits = context @ Wo.T + bo                          # (B, value_vocab)
    probs = softmax(logits, axis=-1)

    cache = {"key_ids": key_ids, "val_ids": val_ids, "X": X, "x_query": x_query,
              "Q": Q, "K": K, "V": V, "scores": scores, "weights": weights,
              "context": context, "probs": probs}
    return logits, weights, cache

## Backward: every step of the forward pass, undone by hand

Every gradient below follows the same three §6-7 identities: delta = blame
flowing backward, dL/dW = delta x<sup>T</sup>, dL/db = delta, dL/dx =
W<sup>T</sup> delta. This is applied five times, once per step, from the
output back to the embeddings.

- **Output + softmax + cross-entropy** collapse to `probs - Y` (the same
  identity as the MLP example — the log in cross-entropy cancels the
  softmax's own derivative). `dWo = delta_logits.T @ context`, `dcontext =
  delta_logits @ Wo`.
- **Step 4 backward** (`context = weights @ V`, a weighted sum over
  positions): `d context / d weights[l] = V[l]` and `d context / d V[l] =
  weights[l]`, so `dweights = einsum("bd,bld->bl", dcontext, V)` and `dV =
  weights[...,None] * dcontext[:,None,:]`.
- **Step 3 backward — softmax over the *sequence* axis, not classes.** This
  is the trickiest step. It is the exact same identity part-1 §5 derives for
  softmax — `s_i (delta_i - sum_k s_k delta_k)` — just applied along
  positions l instead of along output classes: every position's weight
  depends on every *other* position's weight too (they all divide by the
  same normalizer), so the local gradient isn't just `weights * dweights`;
  it has to subtract off the weighted average of `dweights` first:
  ```
  dot     = (weights * dweights).sum(axis=-1, keepdims=True)   # sum_k s_k delta_k, per sample
  dscores = weights * (dweights - dot)                         # s_i (delta_i - dot)
  ```
- **Step 1/5 backward** (`scores = (Q . K_l) / sqrt(d_k)`): `dQ =
  einsum("bl,bld->bd", dscores, K) / sqrt(d_k)`, `dK = dscores[...,None] *
  Q[:,None,:] / sqrt(d_k)`.
- **Step 2/4 backward — the three projections**, back to `dL/dW = delta
  x^T`: `dWq = x_query.T @ dQ`, `dWk` and `dWv` sum over batch *and*
  positions (`einsum("bld,ble->de", X, dK)`). `dX` accumulates blame from
  all three paths — Q only touches the last position, K and V touch every
  position.
- **Embeddings.** `X = E_key[key_ids] + E_val[val_ids] + E_pos`, so blame is
  scattered back with `np.add.at` to whichever rows were actually looked up
  (the same id used more than once accumulates); `E_pos` broadcasts over the
  batch, so its blame is just summed over the batch.

In [5]:
def backward(params, cache, target):
    Wq, Wk, Wv, Wo = params["Wq"], params["Wk"], params["Wv"], params["Wo"]
    X, weights, V, K, Q = cache["X"], cache["weights"], cache["V"], cache["K"], cache["Q"]
    context, probs, x_query = cache["context"], cache["probs"], cache["x_query"]
    key_ids, val_ids = cache["key_ids"], cache["val_ids"]
    B, L, d = X.shape
    d_k = Wq.shape[1]

    # ---- output projection + softmax + cross-entropy (§6-7: they collapse
    # to exactly predicted - actual, same identity as the MLP example) ----
    Y = np.zeros_like(probs)
    Y[np.arange(B), target] = 1
    delta_logits = (probs - Y) / B                        # (B, value_vocab)

    dWo = delta_logits.T @ context                        # dL/dW = delta x^T, summed over batch
    dbo = delta_logits.sum(axis=0)                         # dL/db = delta, summed over batch
    dcontext = delta_logits @ Wo                            # dL/dx = W^T delta (delta is a row, so W on the right)

    # ---- Step 4 backward: context = weights @ V (weighted sum over positions) ----
    # d context[b] / d weights[b,l] = V[b,l];  d context[b] / d V[b,l] = weights[b,l]
    dweights = np.einsum("bd,bld->bl", dcontext, V)         # (B,L)
    dV = weights[:, :, None] * dcontext[:, None, :]          # (B,L,d_v)

    # ---- Step 3 backward: softmax over the SEQUENCE axis (not classes) ----
    # Same identity as §5 (s_i(delta_i - sum_k s_k delta_k)), just applied
    # along positions l instead of along output classes.
    dot = (weights * dweights).sum(axis=-1, keepdims=True)
    dscores = weights * (dweights - dot)                    # (B,L)

    # ---- Step 1/5 backward: scores = (Q . K_l) / sqrt(d_k) ----
    dQ = np.einsum("bl,bld->bd", dscores, K) / np.sqrt(d_k)  # (B,d_k)
    dK = dscores[:, :, None] * Q[:, None, :] / np.sqrt(d_k)   # (B,L,d_k)

    # ---- Step 2/4 backward: the three projections, back to dL/dW = delta x^T ----
    dWq = x_query.T @ dQ                                     # (d, d_k)
    dWk = np.einsum("bld,ble->de", X, dK)                     # (d, d_k), summed over batch AND positions
    dWv = np.einsum("bld,ble->de", X, dV)                     # (d, d_v)

    # dL/dx = W^T delta, accumulated from all three paths (Q only touches the
    # last position; K and V touch every position)
    dX = dK @ Wk.T + dV @ Wv.T                                # (B,L,d)
    dX[:, -1, :] += dQ @ Wq.T

    # ---- embeddings: X = E_key[key_ids] + E_val[val_ids] + E_pos, so blame
    # is scattered back to whichever rows were actually looked up ----
    dE_key = np.zeros_like(params["E_key"])
    np.add.at(dE_key, key_ids, dX)                            # scatter-add: same id used more than once accumulates
    dE_val = np.zeros_like(params["E_val"])
    np.add.at(dE_val, val_ids, dX)
    dE_pos = dX.sum(axis=0)                                    # E_pos broadcasts over the batch -> sum blame over batch

    return {"E_key": dE_key, "E_val": dE_val, "E_pos": dE_pos,
            "Wq": dWq, "Wk": dWk, "Wv": dWv, "Wo": dWo, "bo": dbo}

## Training

Plain minibatch SGD: forward, backward, `params[k] -= lr * grads[k]` for
every trainable array (everything except `meta`). A held-out slice reports
validation loss/accuracy periodically. The `.py` script also persists the
trained weights to `03_weights.npz` via `save_params`/`load_params` so
`predict()` can run later without retraining; that I/O is copied here too but
`train()` takes an explicit `save_path` (default `None`, meaning "don't
save") rather than deriving a path from `__file__`, since a notebook has no
`__file__` — the math is unchanged.

In [6]:
def train(n_pairs=4, key_vocab=6, value_vocab=6, d_model=16, n_samples=4000,
          epochs=60, batch_size=64, lr=0.5, seed=0, verbose=True, save_path=None):
    key_ids, val_ids, targets, match_pos = make_retrieval_task(
        n_pairs=n_pairs, key_vocab=key_vocab, value_vocab=value_vocab,
        n_samples=n_samples, seed=seed)

    # held-out split for reporting
    n_val = max(200, n_samples // 10)
    val_key, val_val, val_targets = key_ids[:n_val], val_ids[:n_val], targets[:n_val]
    train_key, train_val, train_targets = key_ids[n_val:], val_ids[n_val:], targets[n_val:]

    params = init_params(seed=seed, d_model=d_model, n_pairs=n_pairs,
                          key_vocab=key_vocab, value_vocab=value_vocab)
    rng = np.random.default_rng(seed + 1)
    n = train_key.shape[0]
    grad_keys = ["E_key", "E_val", "E_pos", "Wq", "Wk", "Wv", "Wo", "bo"]

    for ep in range(epochs):
        order = rng.permutation(n)
        for s in range(0, n, batch_size):
            idx = order[s:s + batch_size]
            logits, _, cache = forward(train_key[idx], train_val[idx], params)
            grads = backward(params, cache, train_targets[idx])
            for k in grad_keys:
                params[k] -= lr * grads[k]

        if verbose and ((ep + 1) % max(1, epochs // 10) == 0 or ep == epochs - 1):
            val_logits, _, val_cache = forward(val_key, val_val, params)
            val_loss = cross_entropy_loss(val_cache["probs"], val_targets)
            val_acc = (val_logits.argmax(axis=1) == val_targets).mean()
            print(f"epoch {ep + 1:3d}   val loss {val_loss:.4f}   val acc {val_acc:.4f}")

    if save_path is not None:
        save_params(params, save_path)
        if verbose:
            print(f"saved weights to {save_path}")
    return params, (val_key, val_val, val_targets, match_pos[:n_val])


def save_params(params, path):
    meta = params["meta"]
    np.savez(path,
              E_key=params["E_key"], E_val=params["E_val"], E_pos=params["E_pos"],
              Wq=params["Wq"], Wk=params["Wk"], Wv=params["Wv"],
              Wo=params["Wo"], bo=params["bo"],
              meta_d_model=meta["d_model"], meta_n_pairs=meta["n_pairs"],
              meta_key_vocab=meta["key_vocab"], meta_value_vocab=meta["value_vocab"],
              meta_seq_len=meta["seq_len"])


def load_params(path):
    data = np.load(path)
    params = {k: data[k] for k in data.files if not k.startswith("meta_")}
    params["meta"] = {k[len("meta_"):]: int(data[k]) for k in data.files if k.startswith("meta_")}
    return params

## Inference

`encode_raw_sequence` turns a human-readable `[(key,value), ...] + query_key`
into the `(key_ids, val_ids)` arrays `forward` expects, appending the "no
value" sentinel at the query slot. `predict` wraps `forward` for either a
single raw sequence, a batch of raw sequences, or already-encoded arrays, and
returns the predicted value(s) plus the attention weights so they can be
inspected.

In [7]:
def encode_raw_sequence(pairs, query_key, key_vocab, value_vocab):
    """pairs: list of (key,value) ints. query_key: int. Builds (key_ids,
    val_ids) in the id scheme forward() expects, with the sentinel
    "no value" row appended at the query position."""
    n_pairs = len(pairs)
    key_ids = np.empty(n_pairs + 1, dtype=np.int64)
    val_ids = np.empty(n_pairs + 1, dtype=np.int64)
    for i, (k, v) in enumerate(pairs):
        key_ids[i], val_ids[i] = k, v
    key_ids[n_pairs] = query_key
    val_ids[n_pairs] = value_vocab   # sentinel
    return key_ids, val_ids


def predict(token_ids_or_raw_sequence, params_or_path, encode=True):
    """Run inference. When encode=True (default), `token_ids_or_raw_sequence`
    is a raw sequence: either a single (pairs, query_key) tuple where pairs
    is a list of (key,value) ints, or a list of such tuples for a batch.
    When encode=False, it is already-encoded (key_ids, val_ids) as produced
    by make_retrieval_task -- a single pair of (L,) arrays, or a pair of
    (B,L) arrays for a batch. Returns (predicted_value(s), attention_weights)."""
    params = load_params(params_or_path) if isinstance(params_or_path, str) else params_or_path
    meta = params["meta"]
    key_vocab, value_vocab = meta["key_vocab"], meta["value_vocab"]

    if encode:
        seq = token_ids_or_raw_sequence
        single = isinstance(seq, tuple) and len(seq) == 2 and isinstance(seq[0], (list, tuple))
        samples = [seq] if single else seq
        encoded = [encode_raw_sequence(pairs, q, key_vocab, value_vocab) for pairs, q in samples]
        key_ids = np.stack([e[0] for e in encoded])
        val_ids = np.stack([e[1] for e in encoded])
    else:
        key_ids, val_ids = token_ids_or_raw_sequence
        key_ids, val_ids = np.asarray(key_ids), np.asarray(val_ids)
        single = key_ids.ndim == 1
        if single:
            key_ids, val_ids = key_ids[None, :], val_ids[None, :]

    logits, weights, _ = forward(key_ids, val_ids, params)
    preds = logits.argmax(axis=1)
    if single:
        return int(preds[0]), weights[0]
    return preds, weights

## Run it

Same hyperparameters as the `.py` file's `__main__` block: 4 pairs, 6-word
key/value vocabularies, d_model 16, 4000 samples, 60 epochs, batch size 64,
lr 0.5.

In [8]:
N_PAIRS, KEY_VOCAB, VALUE_VOCAB, D_MODEL = 4, 6, 6, 16

params, (val_key, val_val, val_targets, val_match_pos) = train(
    n_pairs=N_PAIRS, key_vocab=KEY_VOCAB, value_vocab=VALUE_VOCAB, d_model=D_MODEL,
    n_samples=4000, epochs=60, batch_size=64, lr=0.5, seed=0)

val_logits, val_weights, _ = forward(val_key, val_val, params)
final_acc = (val_logits.argmax(axis=1) == val_targets).mean()
print(f"\nfinal held-out accuracy: {final_acc:.4f}")

# spot-check: does attention actually land on the matching key?
concentration = val_weights[np.arange(len(val_match_pos)), val_match_pos]
print(f"mean attention weight on the correct pair position: {concentration.mean():.4f}")

epoch   6   val loss 0.0016   val acc 1.0000
epoch  12   val loss 0.0004   val acc 1.0000


epoch  18   val loss 0.0002   val acc 1.0000
epoch  24   val loss 0.0001   val acc 1.0000


epoch  30   val loss 0.0001   val acc 1.0000


epoch  36   val loss 0.0001   val acc 1.0000
epoch  42   val loss 0.0001   val acc 1.0000


epoch  48   val loss 0.0001   val acc 1.0000
epoch  54   val loss 0.0001   val acc 1.0000


epoch  60   val loss 0.0000   val acc 1.0000

final held-out accuracy: 1.0000
mean attention weight on the correct pair position: 0.9613


## Gradient check

The `.py` file was verified this way before being trusted (worst relative
error ~1.75e-6); the check itself isn't checked into the script, so it's run
here directly instead of being gated behind a flag. Central-difference check
of `backward()` against `forward()` alone, for a handful of random entries in
every trainable array (`meta` is skipped — it's not a parameter): perturb one
entry by &plusmn;&epsilon;, take `(loss(+&epsilon;) - loss(-&epsilon;)) /
(2&epsilon;)`, compare against the analytic gradient. This is *not* how the
network trains; it only proves the hand-derived formulas above are correct.

In [9]:
def check_gradients(seed=1, n_checks=3, eps=1e-5):
    rng = np.random.default_rng(seed)
    key_ids, val_ids, targets, _ = make_retrieval_task(
        n_pairs=N_PAIRS, key_vocab=KEY_VOCAB, value_vocab=VALUE_VOCAB, n_samples=8, seed=seed)
    p = init_params(seed=seed, d_model=D_MODEL, n_pairs=N_PAIRS,
                     key_vocab=KEY_VOCAB, value_vocab=VALUE_VOCAB)
    _, _, cache = forward(key_ids, val_ids, p)
    grads = backward(p, cache, targets)

    def loss_at(p):
        _, _, cache = forward(key_ids, val_ids, p)
        return cross_entropy_loss(cache["probs"], targets)

    worst_rel_err = 0.0
    for name in p:
        if name == "meta":
            continue
        arr = p[name]
        shape = arr.shape
        idxs = [tuple(rng.integers(0, d) for d in shape) for _ in range(n_checks)]
        for idx in idxs:
            orig = arr[idx]

            arr[idx] = orig + eps
            loss_plus = loss_at(p)
            arr[idx] = orig - eps
            loss_minus = loss_at(p)
            arr[idx] = orig  # restore

            numeric = (loss_plus - loss_minus) / (2 * eps)
            analytic = grads[name][idx]
            rel_err = abs(numeric - analytic) / max(abs(numeric), abs(analytic), 1e-8)
            worst_rel_err = max(worst_rel_err, rel_err)
            print(f"{name}{idx}: analytic {analytic:+.6e}  numeric {numeric:+.6e}  rel_err {rel_err:.2e}")

    print(f"\nworst relative error: {worst_rel_err:.2e}")
    return worst_rel_err


err = check_gradients()
print("PASS" if err < 1e-4 else "FAIL", "-- hand-derived backward() matches numerical gradients")

E_key(np.int64(2), np.int64(8)): analytic -1.384178e-02  numeric -1.384178e-02  rel_err 6.31e-10
E_key(np.int64(4), np.int64(15)): analytic -1.513201e-02  numeric -1.513201e-02  rel_err 1.12e-09
E_key(np.int64(0), np.int64(2)): analytic +6.331795e-03  numeric +6.331795e-03  rel_err 7.05e-10
E_val(np.int64(5), np.int64(15)): analytic +1.466753e-02  numeric +1.466753e-02  rel_err 8.94e-10
E_val(np.int64(1), np.int64(4)): analytic -1.399011e-02  numeric -1.399011e-02  rel_err 4.88e-10
E_val(np.int64(6), np.int64(6)): analytic -1.978349e-04  numeric -1.978349e-04  rel_err 4.96e-08
E_pos(np.int64(1), np.int64(13)): analytic -2.126143e-02  numeric -2.126143e-02  rel_err 4.51e-10
E_pos(np.int64(1), np.int64(6)): analytic +3.094925e-04  numeric +3.094925e-04  rel_err 6.18e-09
E_pos(np.int64(3), np.int64(8)): analytic -1.572131e-03  numeric -1.572131e-03  rel_err 1.39e-09
Wq(np.int64(1), np.int64(0)): analytic +7.243168e-05  numeric +7.243168e-05  rel_err 3.05e-08
Wq(np.int64(13), np.int64(12))

## Sample predictions

Six fresh held-out sequences (unseen seed), with the attention weights
printed alongside each prediction so you can see them concentrate on the
position whose key matches the query (marked with `^^^^`).

In [10]:
print("--- sample predictions ---")
demo_key, demo_val, demo_targets, demo_match_pos = make_retrieval_task(
    n_pairs=N_PAIRS, key_vocab=KEY_VOCAB, value_vocab=VALUE_VOCAB, n_samples=6, seed=99)

demo_logits, demo_weights, _ = forward(demo_key, demo_val, params)
demo_preds = demo_logits.argmax(axis=1)

for i in range(len(demo_key)):
    keys = demo_key[i, :N_PAIRS]
    values = demo_val[i, :N_PAIRS]
    query_key = demo_key[i, -1]
    w = demo_weights[i]
    pred = demo_preds[i]

    readable = " ".join(f"k{keys[j]}v{values[j]}" for j in range(N_PAIRS)) + f" | Q=k{query_key}"
    bars = " ".join(f"{x:.2f}" for x in w)
    marker_row = " ".join("^^^^" if j == demo_match_pos[i] else "    " for j in range(len(w)))
    correct = "OK" if pred == demo_targets[i] else "WRONG"
    print(f"seq: {readable}")
    print(f"  predicted value = {pred}   actual value = {demo_targets[i]}   [{correct}]")
    print(f"  attn weights   = [{bars}]")
    print(f"  match position = [{marker_row}]  (position {demo_match_pos[i]} is the matching pair)\n")

--- sample predictions ---
seq: k5v1 k4v2 k3v3 k2v4 | Q=k2
  predicted value = 4   actual value = 4   [OK]
  attn weights   = [0.03 0.00 0.00 0.96 0.01]
  match position = [               ^^^^     ]  (position 3 is the matching pair)

seq: k1v0 k0v4 k3v1 k2v3 | Q=k0
  predicted value = 4   actual value = 4   [OK]
  attn weights   = [0.02 0.97 0.00 0.00 0.00]
  match position = [     ^^^^               ]  (position 1 is the matching pair)

seq: k2v2 k5v0 k1v5 k3v3 | Q=k2
  predicted value = 2   actual value = 2   [OK]
  attn weights   = [0.95 0.03 0.00 0.00 0.01]
  match position = [^^^^                    ]  (position 0 is the matching pair)

seq: k4v4 k0v2 k1v5 k3v3 | Q=k1
  predicted value = 5   actual value = 5   [OK]
  attn weights   = [0.01 0.02 0.93 0.03 0.00]
  match position = [          ^^^^          ]  (position 2 is the matching pair)

seq: k0v0 k4v4 k5v1 k2v3 | Q=k4
  predicted value = 4   actual value = 4   [OK]
  attn weights   = [0.01 0.97 0.00 0.01 0.01]
  match positio

## Attention weights, visualized

One bar per sequence position, height = attention weight, for the first demo
sequence above. The bar at the matching key-value pair should dominate.
Skipped automatically if matplotlib isn't installed -- no new dependency is
added for this notebook.

In [11]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available -- skipping visualization")

if HAS_MPL:
    i = 0  # first demo sequence
    w = demo_weights[i]
    L = len(w)
    match = demo_match_pos[i]
    colors = ["#0B8B8B" if j == match else "#8399A8" for j in range(L)]
    labels = [f"k{demo_key[i, j]}v{demo_val[i, j]}" if j < N_PAIRS else f"Q=k{demo_key[i, j]}"
              for j in range(L)]

    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(range(L), w, color=colors, edgecolor="none", width=0.6)
    ax.set_xticks(range(L))
    ax.set_xticklabels(labels)
    ax.set_ylabel("attention weight")
    ax.set_ylim(0, 1)
    ax.set_title("Attention weights at the query position")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    plt.tight_layout()
    plt.show()

matplotlib not available -- skipping visualization
